# Boltzmann Machines

## What is a Boltzmann Machine?

A **Boltzmann Machine (BM)** is a stochastic, energy-based neural network that learns a probability distribution over data. Unlike a feed-forward network, it has **undirected, symmetric connections** and no fixed input-to-output direction. Its units are usually binary: each unit is sampled as either $0$ or $1$, rather than producing only a deterministic activation.

- **Visible units** $v$ represent observed features, such as a user's movie preferences.
- **Hidden units** $h$ capture latent patterns, such as combinations of preferences shared by many users.
- **Weights and biases** determine which configurations of units are compatible. Learning makes observed patterns more probable.

### Energy and probability

For a binary state vector $s$ containing both visible and hidden units, a general BM has energy

$$E(s) = -\sum_i b_i s_i - \sum_{i<j} w_{ij}s_i s_j.$$

Here $b_i$ is a unit's bias and $w_{ij}=w_{ji}$ is the symmetric connection weight; there are no self-connections. At temperature $T=1$, the probability of a state is

$$P(s) = \frac{e^{-E(s)}}{Z}, \qquad Z = \sum_{s'} e^{-E(s')}.$$

**Lower energy means higher probability.** The partition function $Z$ normalizes the probabilities, but summing over every possible state is generally intractable for large networks. The probability of an observed vector is obtained by summing over hidden states: $P(v)=\sum_h P(v,h)$.

### From a general BM to a Restricted Boltzmann Machine

A general BM can connect units within the visible or hidden groups as well as between them. A **Restricted Boltzmann Machine (RBM)** removes all within-group connections, leaving a bipartite graph. This is the model used in this notebook.

```mermaid
graph LR
    subgraph Visible[Visible units: observed preferences]
        v1((v1))
        v2((v2))
        v3((v3))
    end
    subgraph Hidden[Hidden units: latent patterns]
        h1((h1))
        h2((h2))
    end
    v1 --- h1
    v1 --- h2
    v2 --- h1
    v2 --- h2
    v3 --- h1
    v3 --- h2
```

For visible biases $a_i$, hidden biases $b_j$, and weights $W_{ij}$ between visible unit $i$ and hidden unit $j$, the RBM energy is

$$E(v,h) = -\sum_i a_i v_i - \sum_j b_j h_j - \sum_{i,j} v_i W_{ij} h_j.$$

The restriction makes hidden units conditionally independent given visible units, and vice versa. For binary units, the conditional activation probabilities are

$$P(h_j=1\mid v)=\sigma\!\left(b_j+\sum_i W_{ij}v_i\right),$$

$$P(v_i=1\mid h)=\sigma\!\left(a_i+\sum_j W_{ij}h_j\right), \qquad \sigma(x)=\frac{1}{1+e^{-x}}.$$

These are Bernoulli probabilities: sampling a unit is different from simply using its probability as an activation.

### Learning with Contrastive Divergence

Training aims to maximize the likelihood of observed data. The exact weight gradient compares correlations under the data with correlations under the model:

$$\frac{\partial \log P(v)}{\partial W_{ij}} = \mathbb{E}_{P(h\mid v)}[v_i h_j] - \mathbb{E}_{P(v,h)}[v_i h_j].$$

The model expectation is expensive. **Contrastive Divergence (CD-$k$)** approximates it by starting at a training example and performing $k$ alternating Gibbs steps, sampling hidden units from visible units and then visible units from hidden units.

```mermaid
flowchart LR
    A[Observed vector v0] --> B[Sample hidden units]
    B --> C[Sample reconstructed visible units]
    C --> D{Completed k steps?}
    D -->|No| B
    D -->|Yes| E[Compare data and reconstruction correlations]
    E --> F[Update weights and biases]
```

The positive phase reinforces associations supported by the data; the negative phase weakens associations produced by the reconstruction. CD is an approximation, not an exact likelihood gradient, and a low reconstruction error alone does not guarantee a good generative model.

### Connection to movie recommendations

In this notebook, each user's row is a visible vector with one unit per movie. Ratings of 1 or 2 become $0$ (not liked), ratings of 3 or higher become $1$ (liked), and missing ratings are marked as $-1$. **The $-1$ value is a missing-data marker, not a third binary state**; handling missing entries requires masking or another explicit strategy beyond the binary model described above.

Hidden units learn shared preference patterns. Reconstructed visible probabilities can serve as preference scores for unseen movies, though they are not automatically calibrated recommendation probabilities. RBMs are useful for learning latent representations and collaborative filtering, but sampling cost, approximate training, missing-data handling, and cold-start users or items are practical limitations.


In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.parallel
import torch.optim as optim
import torch.utils.data
from torch.autograd import Variable

## Importing the dataset

In [2]:
movies = pd.read_csv('ml-1m/movies.dat', sep = '::', header = None, engine = 'python', encoding = 'latin-1')
users = pd.read_csv('ml-1m/users.dat', sep = '::', header = None, engine = 'python', encoding = 'latin-1')
ratings = pd.read_csv('ml-1m/ratings.dat', sep = '::', header = None, engine = 'python', encoding = 'latin-1')

## Preparing the training set and the test set

In [3]:
training_set = pd.read_csv('ml-100k/u1.base', delimiter = '\t')
training_set = np.array(training_set, dtype = 'int')
test_set = pd.read_csv('ml-100k/u1.test', delimiter = '\t')
test_set = np.array(test_set, dtype = 'int')

## Getting the number of users and movies

In [4]:
nb_users = int(max(max(training_set[:,0]), max(test_set[:,0])))
nb_movies = int(max(max(training_set[:,1]), max(test_set[:,1])))

## Converting the data into an array with users in lines and movies in columns

In [5]:
def convert(data):
    new_data = []
    for id_users in range(1, nb_users + 1):
        id_movies = data[:,1][data[:,0] == id_users]
        id_ratings = data[:,2][data[:,0] == id_users]
        ratings = np.zeros(nb_movies)
        ratings[id_movies - 1] = id_ratings
        new_data.append(list(ratings))
    return new_data
training_set = convert(training_set)
test_set = convert(test_set)

## Converting the data into Torch tensors

In [6]:
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(f'PyTorch device: {device}')
training_set = torch.as_tensor(training_set, dtype=torch.float32, device=device)
test_set = torch.as_tensor(test_set, dtype=torch.float32, device=device)

## Converting the ratings into binary ratings 1 (Liked) or 0 (Not Liked)

In [7]:
training_set[training_set == 0] = -1
training_set[training_set == 1] = 0
training_set[training_set == 2] = 0
training_set[training_set >= 3] = 1
test_set[test_set == 0] = -1
test_set[test_set == 1] = 0
test_set[test_set == 2] = 0
test_set[test_set >= 3] = 1

## Creating the architecture of the Neural Network

In [8]:
class RBM():
    def __init__(self, nv, nh, device):
        self.W = torch.randn(nh, nv, device=device)
        self.a = torch.randn(1, nh, device=device)
        self.b = torch.randn(1, nv, device=device)
    def sample_h(self, x):
        wx = torch.mm(x, self.W.t())
        activation = wx + self.a.expand_as(wx)
        p_h_given_v = torch.sigmoid(activation)
        return p_h_given_v, torch.bernoulli(p_h_given_v)
    def sample_v(self, y):
        wy = torch.mm(y, self.W)
        activation = wy + self.b.expand_as(wy)
        p_v_given_h = torch.sigmoid(activation)
        return p_v_given_h, torch.bernoulli(p_v_given_h)
    def train(self, v0, vk, ph0, phk):
        self.W += torch.mm(v0.t(), ph0).t() - torch.mm(vk.t(), phk).t()
        self.b += torch.sum((v0 - vk), 0)
        self.a += torch.sum((ph0 - phk), 0)
nv = len(training_set[0])
nh = 100
batch_size = 100
rbm = RBM(nv, nh, device)

## Training the RBM

In [9]:
nb_epoch = 10
for epoch in range(1, nb_epoch + 1):
    train_loss = 0
    s = 0.
    for id_user in range(0, nb_users - batch_size, batch_size):
        vk = training_set[id_user:id_user+batch_size]
        v0 = training_set[id_user:id_user+batch_size]
        ph0,_ = rbm.sample_h(v0)
        for k in range(10):
            _,hk = rbm.sample_h(vk)
            _,vk = rbm.sample_v(hk)
            vk[v0<0] = v0[v0<0]
        phk,_ = rbm.sample_h(vk)
        rbm.train(v0, vk, ph0, phk)
        train_loss += torch.mean(torch.abs(v0[v0>=0] - vk[v0>=0]))
        s += 1.
    print('epoch: '+str(epoch)+' loss: '+str(train_loss/s))

epoch: 1 loss: tensor(0.3432)
epoch: 2 loss: tensor(0.2414)
epoch: 3 loss: tensor(0.2480)
epoch: 4 loss: tensor(0.2485)
epoch: 5 loss: tensor(0.2471)
epoch: 6 loss: tensor(0.2487)
epoch: 7 loss: tensor(0.2485)
epoch: 8 loss: tensor(0.2463)
epoch: 9 loss: tensor(0.2457)
epoch: 10 loss: tensor(0.2475)


## Testing the RBM

In [10]:
test_loss = 0
s = 0.
for id_user in range(nb_users):
    v = training_set[id_user:id_user+1]
    vt = test_set[id_user:id_user+1]
    if len(vt[vt>=0]) > 0:
        _,h = rbm.sample_h(v)
        _,v = rbm.sample_v(h)
        test_loss += torch.mean(torch.abs(vt[vt>=0] - v[vt>=0]))
        s += 1.
print('test loss: '+str(test_loss/s))

test loss: tensor(0.2265)
